## 6. Final Assessment & Independent Exercise: Multi-Sensor Alternative Spectral Index & Time-Series Export


🎯 **Your Turn 12 (Final Synthesis Challenge):**

> Create a new Jupyter Notebook in your GitHub repository and replicate what you  consider is necessary to reach waht is required as follows:
>
>  1. Choose one index distinct from NDVI (e.g., **EVI**, **SAVI**, or **NDWI**) for canopy, soil, or moisture evaluation.
> 2.Generate annual median composites for both **Sentinel-2** and **Landsat 8** over your area, applying sensor-specific cloud masking (SCL / QA_PIXEL) and Landsat 8 Surface Reflectance scaling factors.
> 3. **Construct Multi-Temporal Stacks:** Combine the 10 annual index layers (2016 to 2025) into a single 10-band `ee.Image` stack for Sentinel-2 and a corresponding 10-band stack for Landsat 8 (renaming bands sequentially as `Index_2016`, `Index_2017`, ..., `Index_2025`).
> 4. **GeoTIFF Export:** Export both 10-band image stacks to Google Drive as projected GeoTIFF rasters using the official local CRS for Colombia (**EPSG:9377**).

> 5. Commit and push your final `.ipynb` file to your public GitHub repository.
> 6. Ensure your notebook contains executed output cells, structured Markdown headers, and code comments explaining your custom study area selection.
> 7. Copy the direct link to your Jupyter Notebook on GitHub and submit it using the following link:
>
> 📋 [**Submit Your Final Notebook Link Here**](https://forms.gle/73ypriddjvWtarV8A)

---

## Importar todos los paquetes que se van a usar dentro del desarrollo del notebook

In [1]:
import geopandas as gpd # Mnipular dataframes
import rasterio # Manipular raster datasets como arreglos numpy
import rasterio.mask # Modulo de rasterio para cortar raster con formas vector
import rasterio.warp # Modulo de rasterio para automatizar funciones de resampleo 
from rasterio.enums import Resampling # Algoritmos nativos de rasterio de remuestreo o interpolación para resampleo 
import ee # API de google earth engine
import numpy as np # Manipular arreglos
import matplotlib.pyplot as plt # Manipular y crear gráficos
from pathlib import Path # Gestion de rutas de archivos
import geemap # herramientas para mostrar en mapa las imágenes generadas en GEE

## Generar la autenticación de google earth engine para usar la api

In [22]:
ee.Authenticate(
    force=True,
    scopes=[
        'https://www.googleapis.com/auth/earthengine',
        'https://www.googleapis.com/auth/drive'
    ]
)

Enter verification code:  4/1ATsMZqB_pC60pMJY2XHfFjo-WkyPbXDy4Mb6xCS6K4G2fqIrxwISw9DI7Rw



Successfully saved authorization token.


## Inicializar la api y comprobar que funciona bien

In [23]:
try:
    ee.Initialize()
    print("Google Earth Engine initialised successfully.")
except Exception as e:
    print(f"Error initialising GEE: {e}")

Google Earth Engine initialised successfully.


## Dependencias Locales
1. Cargar la dirección del geopackage de municipios de Colombia
2. Leer con geopandas el geopackage de municipios
3. Seleccionar un municipio de interés
4. Establecer el WKT en EPSG 9377 para más adelante exportar las salidas de GEE

In [28]:
root_folder=Path(r"/notebooks/GEOPROCESAMIENTO_UNAL")
gdf = gpd.read_file(root_folder /"municipios_colombia.gpkg")
gdf_muni = gdf[gdf["MPIO_CNMBR"] == "LA MACARENA"].copy()
wkt_9377 = 'PROJCS["MAGNA-SIRGAS / Origen-Nacional", GEOGCS["MAGNA-SIRGAS", DATUM["Marco_Geocentrico_Nacional_de_Referencia", SPHEROID["GRS 1980", 6378137, 298.257222101, AUTHORITY["EPSG","7019"]], AUTHORITY["EPSG","6686"]], PRIMEM["Greenwich", 0, AUTHORITY["EPSG","8901"]], UNIT["degree", 0.0174532925199433, AUTHORITY["EPSG","9122"]], AUTHORITY["EPSG","4686"]], PROJECTION["Transverse_Mercator"], PARAMETER["latitude_of_origin", 4], PARAMETER["central_meridian", -73], PARAMETER["scale_factor", 0.9992], PARAMETER["false_easting", 5000000], PARAMETER["false_northing", 2000000], UNIT["metre", 1, AUTHORITY["EPSG","9001"]], AUTHORITY["EPSG","9377"]]'

## Generar Límites para trabajar en GEE
1. Establecer el bounding box del municipio seleccionado
2. Convertir esa bbox a geojson para que GEE pueda leerlo 

In [5]:
bbox = gdf_muni.to_crs(epsg=4326).total_bounds
ee_bounds = ee.Geometry.BBox(bbox[0], bbox[1], bbox[2], bbox[3])
geojson_geom = gdf_muni.to_crs(epsg=4326).geometry.iloc[0].__geo_interface__
ee_muni_geom = ee.Geometry(geojson_geom)

## Crear un mapa donde se evidencia que todo el proceso de selección quedó bien
1. Se inicializa el GEEMAP
2. Se adiciona el geodataframe del municipio, se establece el nombre y el color del polígono
3. Se adiciona la capa del BBox
4. Se adiciona el JSON que debe ser el mismo que el municipio (corroborar)
5. Se centra el mapa en el centroide del municipio con la función de *centerObject*
6. Se muestra el mapa

In [6]:
Map = geemap.Map()

# 2. Plotear el GeoDataFrame de GeoPandas
Map.add_gdf(gdf_muni, layer_name="La Macarena", fill_colors=["green"])

# 3. Plotear el BBox
Map.addLayer(ee_bounds, {'color': 'red'}, "BBox Municipio")

# 4. Plotear el Polígono (ee_muni_geom)
Map.addLayer(ee_muni_geom, {'color': 'blue'}, "Polígono geoJSON")

# 5. Centrar la vista del mapa en el municipio con un zoom 11
Map.centerObject(ee_muni_geom, zoom=11)

# 6. Mostrar el mapa
Map

Map(center=[2.16185128203109, -74.09487481238008], controls=(WidgetControl(options=['position', 'transparent_b…

## Función de mascara de nubes para Sentinel 2

Se genera la función de máscara de nubes para las imágenes Sentinel 2. La función obtiene la banda SceneClasification, de ahí obtiene las clases de pixeles de sombras, nubes medias
nubes altas y las tipo cirrus, generando una imagen de las mismas dimensiones y los valores de clase donde esos pixeles se enmascaran.

In [7]:
def mask_s2_clouds_scl(image):
    """Masks clouds, cloud shadows, and cirrus using the SCL band."""
    scl = image.select('SCL')
    
    # Identify unwanted pixel classes
    cloud_shadows = scl.eq(3)
    clouds_medium = scl.eq(8)
    clouds_high = scl.eq(9)
    cirrus = scl.eq(10)
    
    # Combine all mask conditions (1 = invalid pixel)
    mask = cloud_shadows.Or(clouds_medium).Or(clouds_high).Or(cirrus).Not()
    
    # Update image mask and retain properties
    return image.updateMask(mask)

## Rutina para obtener el índice NDWI para las imágenes sentinel 2 desde el 2016 hasta el 2025
1. Se establece el rango de años en los que se centra el cálculo
2. Define la función para el cálculo del NDWI anual:
    <ol type="a">
    <li>Obtiene la colección de imágenes para las fechas definidas entre el 1 de enero y el 31 de diciembre de cada año del rango</li>
    <li>Filtra la colección por las imágenes con menos del 30% de nubes</li>
    <li>Enmascara con la función definida previamente</li>
    <li>Obtiene la mediana de la colección</li>
    <li>Se define la variable que va a almacenar los nombres de cada imagen del año concatenando "NDWI_" con el numero del año</li>
    <li>Calcula el NDWI para cada imagen anual obtenida</li>
    <li>Devuelve la imagen calculada y renombrada con el nombre del año</li>
    </ol>
3. Se constuye la colección de imágenes de índices llamando la función definida en el punto 2 con el rango de años definido en el punto 1.
4. Se establecen los nombres de las bandas como función del rango de año, concatenando "NDWI_".
5. Se arma el stack de imágenes NDWI calculadas, como bandas y renombradas con los nombres definidos en el punto 4.

In [8]:

# 1. Rango de años para las imágenes sentinel 2 (2016 to 2025)
years = ee.List.sequence(2016, 2025)

# 2. Funcion para calcular el NDWI para cada año del rango
def compute_annual_ndwi(year):
    date_start = ee.Date.fromYMD(year, 1, 1)
    date_end = ee.Date.fromYMD(year, 12, 31)
    
    annual_s2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ee_muni_geom)
        .filterDate(date_start, date_end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
        .map(mask_s2_clouds_scl)
        .median()
        .clip(ee_muni_geom)
        
    )
    nombre_banda = ee.String("NDWI_").cat(ee.Number(year).format('%04d'))
    
    annual_ndwi = (
        annual_s2
        .normalizedDifference(["B3", "B8"])
        .set("year", year)
        
    )
    
    return annual_ndwi.rename([nombre_banda])

# Construye una colección de los NDWI de cada año
annual_ndwi_collection = ee.ImageCollection(years.map(compute_annual_ndwi))
# Crea la variable que guarda los nombres de las bandas como función de los años en formato de 4 números y los concatena como texto
s2_nombres_bandas = years.map(lambda y: ee.String("NDWI_").cat(ee.Number(y).format('%04d')))
# Crea el Stack de imágenes NDWI
s2_stack = annual_ndwi_collection.toBands().rename(s2_nombres_bandas)

## Mapa para visualizar resultados del calculo de NDWI anual Sentinel 2

In [9]:
# 1. Crear el mapa
Map_ndwiSentinel= geemap.Map()

# 2. Centrar el mapa en la geometría de tu municipio
Map_ndwiSentinel.centerObject(ee_muni_geom, zoom=11)

# 3. Definir los parámetros de visualización para NDWI
ndwi_vis = {
    'min': -0.7,  # Mínimo para ver la vegetación densa
    'max': 0.2,   # Máximo para ver el agua
    'palette': [
        '#002200', # Verde oscuro (Vegetación densa)
        '#006400', # Verde(vegetación normal)
        '#8B4513', # Café (suelos)
        '#FFFFFF', # Blanco (valor 0)
        '#00FFFF', # Azul claro (alta humedad / cuerpos de agua poco profundos)
        '#0000FF'  # Azul (cuerpos de agua profundos)
    ]
}


# 4. Obtener la lista de nombres de las bandas
# .getInfo() extrae la lista de los servidores de Google a tu entorno de Python
nombres_bandas = s2_stack.bandNames().getInfo()

# 5. Bucle para agregar cada banda al mapa
for banda in nombres_bandas:
    # Selecciona solo la banda actual y la agrega con los colores definidos
    Map_ndwiSentinel.addLayer(s2_stack.select(banda), ndwi_vis, banda)

# 6. Mostrar el mapa final en la celda
Map_ndwiSentinel

Map(center=[2.1618512820310674, -74.09487481238014], controls=(WidgetControl(options=['position', 'transparent…

## Exportar stack Sentinel 2
1. Se establece la tarea de exportar el stack de bandas definido en la sección anterior.
2. Se definen los argumentos de la función de exportar nativa de GEE: el stack, la descripción, el nombre de la carpeta en el folder (si no existe, lo crea), la región es la zona de estudio (el municipio), la escala (resolución espacial), el sistema coordenado de referencia con base en el WKT definido previamente y una ventana de pixeles con un límite que evita la saturación de GEE en exportar.

In [31]:
# Exportar la colección de imágenes a Google Drive

task_s2 = ee.batch.Export.image.toDrive(
    image=s2_stack,
    description='Sentinel2_NDWI_Stack_2016_2025',
    folder='GEOPROCESAMIENTO_UNAL', 
    region=ee_muni_geom,
    scale=10, 
    crs=wkt_9377,
    maxPixels=1e13
)
task_s2.start()

print("Exportado Stack Sentinel 2")

Exportado Stack Sentinel 2


## Función para máscara de nubes en Landsat
La Colección Landsat 8, tiene una capa llamada QA que funciona como el "interruptor": indica si hay nubes está en la posición del bit 3, y el de las sombras de nubes está en el bit 4, entonces, para sombras:

1. Se hace una operación de desplazamiento a la izquierda. Toma un 1 en binario y lo mueve 4 posiciones (00010000). Esto crea una plantilla para apuntar solo al bit 4 de la información del píxel.
2. Superpone la información del bit 4 sobre el valor binario del píxel original de la imagen. Si el píxel original tiene el bit 4 con informacón, es decir, el algoritmo de Landsat detectó una sombra; la operación arroja un valor distinto de cero. Si no tiene sombra genera un valor de 0.
3. Se queda solo con los píxeles donde el valor extraído sea exactamente igual a 0 (donde NO hay sombra)

La función de mascara de nubes hace el mismo procedimiento pero estableciendo el desplazamiento del binario 3 posiciones, revisa si ese bit tiene información y se queda solo con el pixel donde ese valor es 0. Retorna la imagen con la máscara de nubes y sombras, cruzando ambas máscaras booleanas. Para que un píxel se seleccionado debe cumplir las dos condiciones: No tener sombra Y no tener nube.

In [10]:
def mask_landsat8_clouds(image):
 # Mascara de nubes y sombras usando la banda QA_PIXEL de Landsat 8
    qa = image.select('QA_PIXEL')
    # Bits 3 y 4 representan nubes y sombras de nubes respectivamente
    cloud_shadow = qa.bitwiseAnd(1 << 4).eq(0)
    clouds = qa.bitwiseAnd(1 << 3).eq(0)
    return image.updateMask(cloud_shadow.And(clouds))

## Rutina para obtener el índice NDWI para las imágenes Landsat 8 desde el 2016 hasta el 2025
1. Se define la función para pasar los valores digitales de la imagen Landsat a valores de reflectancia
2. Se establece el rango de años en los que se centra el cálculo
3. Define la función para el cálculo del NDWI anual:
    <ol type="a">
    <li>Obtiene la colección de imágenes para las fechas definidas entre el 1 de enero y el 31 de diciembre de cada año del rango</li>
    <li>Filtra la colección por las imágenes con menos del 30% de nubes</li>
    <li>Enmascara con la función definida previamente</li>
    <li>Se aplica el factor de escala para pasar los valores digitales a reflectancia</li>
    <li>Se define la variable que va a almacenar los nombres de cada imagen del año concatenando "NDWI_" con el numero del año</li>
    <li>Obtiene la mediana de la colección</li>
    <li>Calcula el NDWI para cada imagen anual obtenida</li>
    <li>Devuelve la imagen calculada y renombrada con el nombre del año</li>
    </ol>
4. Se constuye la colección de imágenes de índices llamando la función definida en el punto 2 con el rango de años definido en el punto 1.
5. Se establecen los nombres de las bandas como función del rango de año, concatenando "NDWI_".
6. Se arma el stack de imágenes NDWI calculadas, como bandas y renombradas con los nombres definidos en el punto 4.

In [11]:
# Escalado para pasar de valores digitales a valores de reflectancia para Landsat 8
def scale_landsat8(image):
    optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    return image.addBands(optical_bands, None, True)

years = ee.List.sequence(2016, 2025)

# 2. Funcion para calcular el NDWI para cada año del rango
def compute_annual_ndwi_land8(year):
    date_start = ee.Date.fromYMD(year, 1, 1)
    date_end = ee.Date.fromYMD(year, 12, 31)
    
    landsat8_collection = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(ee_muni_geom)
        .filterDate(date_start, date_end)
        .filter(ee.Filter.lt("CLOUD_COVER", 30))
        .map(mask_landsat8_clouds) 
        .map(scale_landsat8)
        
)

    nombre_banda_landsat8 = ee.String("NDWI_").cat(ee.Number(year).format('%04d'))
    
    annual_ndwi_land8 = ( landsat8_collection
        .median()
        .clip(ee_muni_geom)
        .normalizedDifference(['SR_B3', 'SR_B5'])
        .set("year", year)
    )
    
    return annual_ndwi_land8.rename([nombre_banda_landsat8])

# Construye una colección de los NDWI para Landsat de cada año
annual_ndwi_collection = ee.ImageCollection(years.map(compute_annual_ndwi_land8))
# Crea la variable que guarda los nombres de las bandas como función de los años en formato de 4 números y los concatena como texto
landsat8_nombres_bandas = years.map(lambda y: ee.String("NDWI_").cat(ee.Number(y).format('%04d')))
# Crea el Stack de imágenes NDWI
l8_stack = annual_ndwi_collection.toBands().rename(landsat8_nombres_bandas)

## Mapa para visualizar resultados del calculo de NDWI anual Landsat 8

In [21]:
# 1. Crear el mapa
Map_ndwiLandsat= geemap.Map()

# 2. Centrar el mapa en la geometría de tu municipio
Map_ndwiLandsat.centerObject(ee_muni_geom, zoom=11)

# 3. Definir los parámetros de visualización específicos para NDWI
ndwi_vis_l8 = {
    'min': -0.7,
    'max': 0.3,
    'palette': [
        '#002200', # Verde oscuro (Vegetación densa)
        '#006400', # Verde (vegetación media)
        '#8B4513', # Cafe (suelos)
        '#FFFFFF', # Blanco (valor 0)
        '#00FFFF', # Azul claro (alta humedad / cuerpos de agua poco profundos)
        '#0000FF'  # Azul (Cuerpos de agua profundos)
    ]
}


# 4. Obtener la lista de nombres de las bandas
# .getInfo() extrae la lista de los servidores de Google a tu entorno de Python
nombres_bandas = s2_stack.bandNames().getInfo()

# 5. Bucle para agregar cada banda al mapa
for banda in nombres_bandas:
    # Selecciona solo la banda actual y la agrega con los colores definidos
    Map_ndwiLandsat.addLayer(l8_stack.select(banda), ndwi_vis_l8, banda)

# 6. Mostrar el mapa final en la celda
Map_ndwiLandsat

Map(center=[2.1618512820310674, -74.09487481238014], controls=(WidgetControl(options=['position', 'transparent…

## Exportar stack Landsat 8
1. Se establece la tarea de exportar al drive el stack de bandas definido en la sección anterior.
2. Se definen los argumentos de la función de exportar nativa de GEE: el stack, la descripción, el nombre de la carpeta en el folder (si no existe, lo crea), la región es la zona de estudio (el municipio), la escala (resolución espacial), el sistema coordenado de referencia con base en el WKT que se definió previamente y una ventana de pixeles con un límite que evita la saturación de GEE en exportar.

In [29]:
# Exportar la colección de imágenes a Google Drive
task_l8 = ee.batch.Export.image.toDrive(
    image=l8_stack,
    description='Landsat8_NDWI_Stack_2016_2025',
    folder='GEOPROCESAMIENTO_UNAL',
    region=ee_muni_geom,
    scale=30,
    crs=wkt_9377,
    maxPixels=1e13
)
task_l8.start()
print("Exportado Stack Landsat 8")

Exportado Stack Landsat 8


In [ ]:
# Consultar las exportaciones enviadas al google drive
tareas = ee.batch.Task.list()

print("=== Monitor de Exportaciones - Google Drive ===")

# Se inicia el for para revisar las 5 primeras tareas enviadas a drive
for tarea in tareas[:5]: 
    # Se obtiene la información de cada tarea
    info_tarea = tarea.status() 
    
    descripcion = info_tarea.get('description', 'Sin descripción')
    estado = info_tarea.get('state', 'UNKNOWN')
    
    print(f" Archivo: {descripcion}")
    print(f" Estado: {estado}")
    
    # Condicional if para saber cual es el error
    if estado == 'FAILED':
        error = info_tarea.get('error_message', 'No hay detalles del error')
        print(f" Cual es el error: {error}")
        
    print("-------------------------------------------")

# Ejecutar manualmente hasta que ambas tareas pasen su estado a terminado

=== Monitor de Exportaciones - Google Drive ===
 Archivo: Sentinel2_NDWI_Stack_2016_2025
 Estado: COMPLETED
-------------------------------------------
 Archivo: Landsat8_NDWI_Stack_2016_2025
 Estado: COMPLETED
-------------------------------------------
